# MicroDuck training on Google Colab
Open this notebook in Colab, choose a GPU runtime, then run the cell. GPU access and compute-unit use depend on your account. Results go to `MyDrive/microduck/`, and the runtime is unassigned when the cell ends, so nothing keeps billing after training. Set `auto_release = False` to keep it.

在 Colab 中打开此 Notebook，选择 GPU 运行时并运行代码。可用 GPU 与算力消耗由您的账号决定。结果保存到 Google Drive 的 `MyDrive/microduck/`，代码结束后会自动断开运行时，避免空转扣费；如需保留运行时，请设置 `auto_release = False`。

In [ ]:
# ---- 按需修改 / edit as needed ----
task = 'Mjlab-Velocity-Flat-MicroDuck'
iterations = 1000
envs = 4096
max_hours = 3              # 时长上限：到时自动停止训练 / wall-clock limit for training
auto_release = True        # 结束后自动断开运行时，避免空转扣费 / unassign when finished
revision = 'cb70b792312d559a4da09064d92009079671815f'  # 固定官方源码版本 / pinned upstream
# ----------------------------------
import os, pathlib, shutil, subprocess, sys, time
from google.colab import drive, runtime

# 先挂载 Drive：结果存入 Drive 才能在自动断开后保留。Mount first: only Drive survives the unassign.
drive.mount('/content/drive')
out = pathlib.Path('/content/drive/MyDrive/microduck') / time.strftime('%Y%m%d-%H%M%S')
out.mkdir(parents=True, exist_ok=True)
repo = pathlib.Path('/content/microduck_rl')

def save_latest():
    checkpoints = sorted(repo.glob('logs/**/model_*.pt'), key=lambda p: p.stat().st_mtime)
    if not checkpoints:
        print('尚无检查点 / no checkpoint yet')
        return None
    shutil.copy2(checkpoints[-1], out / 'checkpoint.pt')
    return checkpoints[-1]

try:
    subprocess.run(['nvidia-smi'], check=True)  # 没有 GPU 时立即失败 / fail fast without a GPU
    if not shutil.which('uv'):
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/pollen-robotics/microduck_rl.git', str(repo)], check=True)
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', revision], cwd=repo, check=True)
    subprocess.run(['git', 'checkout', '--detach', revision], cwd=repo, check=True)
    subprocess.run(['uv', 'sync', '--no-dev'], cwd=repo, check=True)
    timed_out = False
    try:
        subprocess.run(['uv', 'run', 'train', task, '--env.scene.num-envs', str(envs), '--agent.max_iterations', str(iterations)],
                       cwd=repo, env={**os.environ, 'WANDB_MODE': 'disabled'}, check=True, timeout=max_hours * 3600)
    except subprocess.TimeoutExpired:
        timed_out = True
        print(f'已达到 {max_hours} 小时上限，保存最新检查点 / hit the time limit, saving the latest checkpoint')
    checkpoint = save_latest()
    if checkpoint is not None:
        subprocess.run(['uv', 'run', 'python', 'scripts/export.py', task, '--checkpoint-file', str(checkpoint), '--num-envs', '1', '--onnx-file', str(out / 'policy.onnx')],
                       cwd=repo, check=True, timeout=1800)
    print('结果已保存到 / saved to', out, '(timed out)' if timed_out else '')
except BaseException:
    save_latest()  # 失败或手动中断也尽量保留进度 / keep progress on failure or interrupt
    raise
finally:
    drive.flush_and_unmount()
    if auto_release:
        print('正在断开运行时 / unassigning the runtime')
        runtime.unassign()
